# HyperParameter Tunning (XGBC)

In [1]:
# Imports

import os
import sys
import optuna
import pandas as pd
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split , RandomizedSearchCV , cross_val_score , StratifiedKFold
from scipy.stats import uniform , randint
from sklearn.metrics import f1_score , accuracy_score
from sklearn.metrics import roc_auc_score


sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath(".."))

from preprocessing import get_X_y

C:\Users\sailj\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Data Load

df = pd.read_csv(r"../dataset/train.csv")

df

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699630,699630,57,612,2,1,1,1,1,3,3,...,2,2,4,0,0.0,Female,Loyal Customer,Business travel,Eco,False
699631,699631,30,862,2,2,2,3,4,2,4,...,4,2,4,24,34.0,Female,disloyal Customer,Business travel,Eco,False
699632,699632,49,3148,5,5,5,5,4,4,4,...,5,5,5,0,0.0,Male,Loyal Customer,Business travel,Business,True
699633,699633,61,679,5,2,2,2,5,4,4,...,5,5,1,0,0.0,Female,Loyal Customer,Business travel,Eco,True


In [3]:
# Preprocessing

X_train , y_train = get_X_y(df)

X_train

,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Gender,Customer Type,Type of Travel,Class,Total Delay
0,36,694,4,4,4,3,1,4,1,1,3,4,4,3,1,Female,Loyal Customer,Personal Travel,Eco,0.0
1,56,651,5,5,5,5,5,4,4,5,5,5,5,4,3,Male,Loyal Customer,Business travel,Business,0.0
2,31,1371,3,4,2,5,3,3,3,3,4,3,4,5,3,Female,Loyal Customer,Personal Travel,Eco,0.0
3,10,1616,3,5,3,3,3,3,3,3,4,5,3,4,3,Male,Loyal Customer,Personal Travel,Eco,0.0
4,23,481,3,4,3,4,5,3,5,5,5,4,5,3,5,Female,disloyal Customer,Business travel,Eco,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699630,57,612,2,1,1,1,1,3,3,2,2,2,2,2,4,Female,Loyal Customer,Business travel,Eco,0.0
699631,30,862,2,2,2,3,4,2,4,4,2,5,4,2,4,Female,disloyal Customer,Business travel,Eco,58.0
699632,49,3148,5,5,5,5,4,4,4,5,5,5,5,5,5,Male,Loyal Customer,Business travel,Business,0.0
699633,61,679,5,2,2,2,5,4,4,5,4,5,5,5,1,Female,Loyal Customer,Business travel,Eco,0.0


In [4]:
# Data Prep

cat_cols = X_train.select_dtypes(include = ["string" , "object"]).columns

preprocess = ColumnTransformer([
    ("cat" , OneHotEncoder(handle_unknown = "ignore") , cat_cols)
] , remainder = "passthrough")

model = Pipeline([
    ("preprocess" , preprocess) ,
    ("xgbc" , XGBClassifier())
])

In [45]:
def objective(trial) :

    params = {

        "xgbc__n_estimators" : trial.suggest_int(
            "n_estimators" ,
            200 ,
            2000
        ) ,

        "xgbc__max_depth" : trial.suggest_int(
            "max_depth" ,
            2 , 
            12
        ) ,

        "xgbc__learning_rate" : trial.suggest_float(
            "learning_rate" ,
            0.005 ,
            0.3 ,
            log = True
        ) ,

        "xgbc__min_child_weight" : trial.suggest_int(
            "min_child_weight" ,
            1 ,
            30
        ) ,

        "xgbc__gamma" : trial.suggest_float(
            "gamma" ,
            1e-8 ,
            10 , 
            log = True
        ) ,

        "xgbc__reg_alpha" : trial.suggest_float(
            "reg_alpha" ,
            1e-8 ,
            100 ,
            log = True
        ) ,

        "xgbc__reg_lambda" : trial.suggest_float(
            "reg_lambda" ,
            1e-8 , 
            100 ,
            log = True
        ) ,

        "xgbc__subsample" : trial.suggest_float(
            "subsample" ,
            0.6 , 
            1.0
        ) ,

        "xgbc__colsample_bytree" : trial.suggest_float(
            "colsample_bytree" ,
            0.6 ,
            1.0 
        ) ,
        
        "xgbc__eval_metric" : "auc" ,
        "xgbc__objective"  : 'binary:logistic' , 
        "xgbc__n_jobs" : 1 ,
        "xgbc__random_state" : 42
    }

    model.set_params(**params)

    cv = StratifiedKFold(
        n_splits = 3 ,
        shuffle = True ,
        random_state = 42
    )

    scores = cross_val_score(
        model ,
        X_train ,
        y_train ,
        cv = cv ,
        scoring = "roc_auc" ,
        n_jobs = -1
    )

    return scores.mean()

study = optuna.create_study(
    study_name = "Study 1" ,
    direction = "maximize"
)

study.optimize(
    objective ,
    n_trials = 30
)

print("Best CV ROC-AUC:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-10-09 00:32:55,866] A new study created in memory with name: Study 1
[I 2026-10-09 00:33:44,120] Trial 0 finished with value: 0.9550002218435187 and parameters: {'n_estimators': 613, 'max_depth': 3, 'learning_rate': 0.03119854852818446, 'min_child_weight': 16, 'gamma': 1.827370577473892, 'reg_alpha': 5.078682909856213, 'reg_lambda': 0.018527480502598278, 'subsample': 0.669008652726127, 'colsample_bytree': 0.8212834980354242}. Best is trial 0 with value: 0.9550002218435187.
[I 2026-10-09 00:35:17,647] Trial 1 finished with value: 0.9582006912947053 and parameters: {'n_estimators': 520, 'max_depth': 12, 'learning_rate': 0.04458577734011148, 'min_child_weight': 23, 'gamma': 3.3767826929635685e-05, 'reg_alpha': 8.361237069149528e-05, 'reg_lambda': 2.164313018123939e-08, 'subsample': 0.6428652185208442, 'colsample_bytree': 0.7621621152676328}. Best is trial 1 with value: 0.9582006912947053.
[I 2026-10-09 00:35:59,253] Trial 2 finished with value: 0.9563160604878168 and parameters: {

Best CV ROC-AUC: 0.9587184950167749
Best Parameters:
{'n_estimators': 1528, 'max_depth': 10, 'learning_rate': 0.016049793769125548, 'min_child_weight': 30, 'gamma': 0.00628289414096616, 'reg_alpha': 1.4182527083521002e-08, 'reg_lambda': 63.67109063007865, 'subsample': 0.9192580300880404, 'colsample_bytree': 0.9187012880525618}


In [5]:
# 1st Try

def objective(trial) :

    params = {

        "xgbc__n_estimators" : trial.suggest_int(
            "n_estimators" ,
            200 ,
            2000
        ) ,

        "xgbc__max_depth" : trial.suggest_int(
            "max_depth" ,
            2 , 
            12
        ) ,

        "xgbc__learning_rate" : trial.suggest_float(
            "learning_rate" ,
            0.005 ,
            0.3 ,
            log = True
        ) ,

        "xgbc__min_child_weight" : trial.suggest_int(
            "min_child_weight" ,
            1 ,
            30
        ) ,

        "xgbc__gamma" : trial.suggest_float(
            "gamma" ,
            1e-8 ,
            10 , 
            log = True
        ) ,

        "xgbc__reg_alpha" : trial.suggest_float(
            "reg_alpha" ,
            1e-8 ,
            100 ,
            log = True
        ) ,

        "xgbc__reg_lambda" : trial.suggest_float(
            "reg_lambda" ,
            1e-8 , 
            100 ,
            log = True
        ) ,

        "xgbc__subsample" : trial.suggest_float(
            "subsample" ,
            0.6 , 
            1.0
        ) ,

        "xgbc__colsample_bytree" : trial.suggest_float(
            "colsample_bytree" ,
            0.6 ,
            1.0 
        ) ,
        
        "xgbc__eval_metric" : "auc" ,
        "xgbc__objective"  : 'binary:logistic' , 
        "xgbc__n_jobs" : 1 ,
        "xgbc__random_state" : 42
    }

    model.set_params(**params)

    cv = StratifiedKFold(
        n_splits = 3 ,
        shuffle = True ,
        random_state = 42
    )

    scores = cross_val_score(
        model ,
        X_train ,
        y_train ,
        cv = cv ,
        scoring = "roc_auc" ,
        n_jobs = -1
    )

    return scores.mean()

study = optuna.create_study(
    study_name = "Study 1" ,
    direction = "maximize"
)

study.optimize(
    objective ,
    n_trials = 30
)

print("Best CV ROC-AUC:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-10-10 20:10:17,711] A new study created in memory with name: Study 1
[I 2026-10-10 20:11:26,068] Trial 0 finished with value: 0.9572138207950248 and parameters: {'n_estimators': 905, 'max_depth': 6, 'learning_rate': 0.168910508162265, 'min_child_weight': 4, 'gamma': 8.371635644913104e-07, 'reg_alpha': 5.482191785113308e-07, 'reg_lambda': 0.6418491063285837, 'subsample': 0.8688879754924974, 'colsample_bytree': 0.927153735052281}. Best is trial 0 with value: 0.9572138207950248.
[I 2026-10-10 20:15:30,593] Trial 1 finished with value: 0.9578690564863805 and parameters: {'n_estimators': 1094, 'max_depth': 11, 'learning_rate': 0.023233937741175534, 'min_child_weight': 4, 'gamma': 0.0019634924517776295, 'reg_alpha': 6.865830945849653e-08, 'reg_lambda': 5.488780807737366e-07, 'subsample': 0.638209381041302, 'colsample_bytree': 0.7088382504048041}. Best is trial 1 with value: 0.9578690564863805.
[I 2026-10-10 20:18:00,904] Trial 2 finished with value: 0.958194998507185 and parameters: 

Best CV ROC-AUC: 0.958910926141265
Best Parameters:
{'n_estimators': 1423, 'max_depth': 11, 'learning_rate': 0.013735365034512911, 'min_child_weight': 13, 'gamma': 1.9999543167310362e-06, 'reg_alpha': 6.906442415150357, 'reg_lambda': 9.5312610099552e-05, 'subsample': 0.9260074439454506, 'colsample_bytree': 0.7104877345208437}
